In [1]:
from spot_env import SpotEnv
import numpy as np
from gymnasium.wrappers import FlattenObservation
from stable_baselines3 import SAC, TD3, DDPG, PPO
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv
from sb3_logging import QValueCallback
import pandas as pd

import os

In [2]:
env = DummyVecEnv([lambda: FlattenObservation(SpotEnv())])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_reward=10.0)

In [3]:
timestamps = [
    ]

choose_model = -1 

In [4]:
plot_latest_run = False if 0 < choose_model <= len(timestamps) else True


# List all files in the directory
models = [f for f in os.listdir(r"models\sac_spot_model") 
         if os.path.isfile(os.path.join(r"models\sac_spot_model", f))]

print(models)

if plot_latest_run:
    timestamp = sorted(models)[choose_model]

model_dir = rf'C:\Users\kenfe\PycharmProjects\Spot_Market_Bidding\models\sac_spot_model\{timestamp}'
model = SAC.load(model_dir, env=env)

print(f"Loaded model from {model_dir}")

['2025-01-24T00-37-24.zip', '2025-01-24T18-13-15.zip', '2025-01-24T20-58-24.zip', '2025-01-27T17-53-28.zip', '2025-01-28T10-41-32.zip', '2025-06-27T15-10-20.zip', '2025-07-02T15-19-42.zip']
Loaded model from C:\Users\kenfe\PycharmProjects\Spot_Market_Bidding\models\sac_spot_model\2025-07-02T15-19-42.zip


In [5]:
obs_log = []
reward_log = []

# Run a few episodes and collect data
obs = env.reset()
for _ in range(50):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    obs_log.append(obs.flatten())  # Flatten if it's an array
    reward_log.append(reward)
    if done:
        obs = env.reset()

# Convert to DataFrame
df = pd.DataFrame(obs_log, columns=[f"feature_{i}" for i in range(len(obs.flatten()))])
df["reward"] = reward_log

# Compute correlation
correlations = df.corr()["reward"].sort_values()

Set parameter Username
Academic license - for non-commercial use only - expires 2026-06-13


In [6]:
df

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,reward
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.999950,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,[0.0]
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.336324,0.000000,0.0,1.414249,1.414249,0.000000,0.000000,0.000000,[-2.0]
2,1.732080,1.732080,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.432096,1.732080,0.0,1.609932,0.750917,0.000000,0.000000,0.000000,[-1.733809]
3,1.224765,1.224765,2.000025,2.000025,2.000025,2.000025,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.749449,1.224765,0.0,1.240080,0.905200,0.000000,0.000000,0.000000,[-0.10407172]
4,0.947817,2.209424,1.368245,2.228489,1.306686,-0.229896,2.236090,2.236090,0.000000,0.000000,...,2.236090,1.767359,1.000017,0.0,1.201410,1.406513,2.236090,0.000000,0.000000,[-0.21506467]
5,0.736544,0.042647,1.110448,2.213160,1.068231,0.851051,1.546970,2.202244,0.000000,0.000000,...,2.445503,1.810560,1.460445,0.0,0.533849,0.669940,2.445599,0.000000,0.000000,[0.6849784]
6,0.117967,2.167158,0.878093,0.031936,0.918609,1.825577,1.252704,2.598212,2.645770,2.645770,...,2.303899,1.570424,1.213859,0.0,-0.568652,0.557193,2.331011,0.000000,0.000000,[1.335421]
7,-0.496537,1.523579,-0.286754,-0.116988,0.845467,0.371739,1.108302,0.069004,-0.353551,-0.353551,...,1.722745,2.358748,1.060887,0.0,-0.362428,0.905902,1.735414,0.000000,0.000000,[-0.14773475]
8,-0.301840,-0.399873,-0.052091,0.166106,0.774864,0.350251,1.413591,2.866416,-0.333331,-0.333331,...,1.435189,2.006245,0.954182,0.0,-0.435873,0.612958,1.443307,0.000000,0.000000,[0.081056714]
9,-0.286609,-0.378523,-0.088430,2.909134,-0.301488,-0.165098,0.785665,0.504328,2.426142,3.157469,...,1.255862,1.668851,0.874314,0.0,-0.287183,0.854908,1.261837,0.000000,0.000000,[-0.10513028]


In [7]:
correlations

feature_30   -0.429325
feature_13   -0.329944
feature_0    -0.244741
feature_31   -0.218881
feature_25   -0.190097
feature_1    -0.107702
feature_18   -0.004214
feature_20    0.002581
feature_14    0.012780
feature_27    0.016031
feature_22    0.016550
feature_23    0.021464
feature_16    0.021964
feature_19    0.022994
feature_17    0.023104
feature_11    0.027191
feature_4     0.032761
feature_21    0.040124
feature_10    0.045139
feature_28    0.054921
feature_6     0.059454
feature_15    0.066071
feature_5     0.072764
feature_33    0.109744
feature_34    0.116528
feature_2     0.146697
feature_7     0.148719
feature_3     0.158887
feature_8     0.188337
feature_24    0.190083
feature_9     0.193780
feature_26    0.208171
feature_32    0.228332
feature_12    0.253343
reward        1.000000
feature_29         NaN
Name: reward, dtype: float64